[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrcsvg/ufpr-ppgcd-acidentes-no-trabalho/blob/claude/novo-projeto-o7g3f2/notebooks/01_consolidacao_e_eda.ipynb)

# Acidentes de trabalho no Brasil — consolidação e análise exploratória

**PPGCD/UFPR** · microdados de **CAT** (Comunicação de Acidente de Trabalho)

Este notebook percorre o caminho completo: sai de **61 arquivos CSV que não conversam
entre si** e chega a uma **base única de 3,47 milhões de registros**, mostrando ao vivo
cada armadilha encontrada no caminho e como ela foi resolvida.

A ênfase está no **processo de consolidação**. Boa parte das decisões tomadas aqui não
aparece no resultado final, mas muda os números — em alguns casos, muda bastante.

| Etapa | O que acontece |
|---|---|
| 1 | Reconhecer o acervo: 61 arquivos, 5 cabeçalhos diferentes |
| 2 | As armadilhas visíveis já no cabeçalho |
| 3 | Rodar o pipeline de consolidação |
| 4 | As armadilhas que só aparecem na base inteira |
| 5 | Análise exploratória |
| 6 | Síntese e limitações |

> ⏱️ O notebook inteiro roda em torno de **5 a 8 minutos** no Colab. A etapa mais
> demorada é baixar 1,8 GB de CSV.

Código, testes e documentação: [github.com/mrcsvg/ufpr-ppgcd-acidentes-no-trabalho](https://github.com/mrcsvg/ufpr-ppgcd-acidentes-no-trabalho)


## 0. Preparação

Clonamos o repositório e instalamos o pacote. Clonar (em vez de instalar direto do GitHub) faz os caminhos do projeto — `data/raw`, `data/processed` — resolverem corretamente.


In [ ]:
!git clone --quiet --depth 1 --branch claude/novo-projeto-o7g3f2 https://github.com/mrcsvg/ufpr-ppgcd-acidentes-no-trabalho.git
%cd ufpr-ppgcd-acidentes-no-trabalho
!pip install --quiet -e .
print("pronto")


In [ ]:
import os

# O acervo está em um bucket público do Google Cloud Storage.
os.environ["BUCKET_CAT"] = "acidentes-no-trabalho"

import pandas as pd
from IPython.display import Image, display

from acidentes_trabalho import figuras, pipeline
from acidentes_trabalho.config import DADOS_RAW
from acidentes_trabalho.dados import datas, dicionario, esquemas, gcs, limpeza

# num() e pct() formatam no padrão brasileiro: 3.473.749 e 11,7%
from acidentes_trabalho.relatorio import num, pct

figuras.aplicar_estilo()
pd.set_option("display.max_colwidth", 60)
print(f"pandas {pd.__version__}")


## 1. O acervo

Primeiro, o que existe no bucket. A listagem usa a API pública do GCS — não baixa nada,
só pergunta o que há lá.


In [ ]:
objetos = gcs.listar(prefixo="cats/")
total_gb = sum(o.tamanho for o in objetos) / 1e9

print(f"{len(objetos)} arquivos · {total_gb:.2f} GB\n")
for objeto in sorted(objetos, key=lambda o: -o.tamanho)[:5]:
    print(f"  {objeto.tamanho/1e6:7.1f} MB  {objeto.nome}")
print("  ...")


Os nomes seguem duas convenções — `D.SDA.PDA.005.CAT.AAAAMM.csv` para 2021 em diante e
`cat-comp01-02-03-2020.csv` para os mais antigos. Isso já sugere que o acervo foi montado
em momentos diferentes.

### 1.1 Cinco cabeçalhos diferentes

A pergunta que decide todo o resto: **os arquivos têm o mesmo formato?**

Para responder sem baixar 1,8 GB, pedimos ao servidor apenas os primeiros bytes de cada
arquivo — uma requisição HTTP com cabeçalho `Range`. Sessenta e um pedidos de 3 KB.


In [ ]:
import collections
import urllib.parse
import urllib.request


def primeira_linha(nome, bytes_=3000):
    """Baixa só o início do arquivo e devolve a linha de cabeçalho."""
    url = f"https://storage.googleapis.com/acidentes-no-trabalho/{urllib.parse.quote(nome)}"
    pedido = urllib.request.Request(url, headers={"Range": f"bytes=0-{bytes_}"})
    with urllib.request.urlopen(pedido, timeout=60) as resposta:
        return resposta.read().decode("latin-1").split("\n")[0].strip()

cabecalhos = collections.defaultdict(list)
for objeto in objetos:
    cabecalhos[primeira_linha(objeto.nome)].append(objeto.nome)

print(f"{len(objetos)} arquivos → {len(cabecalhos)} cabeçalhos DIFERENTES\n")
for i, (cabecalho, arquivos) in enumerate(
    sorted(cabecalhos.items(), key=lambda kv: -len(kv[1])), 1
):
    colunas = cabecalho.split(";")
    print(f"esquema {i}: {len(colunas):>2} colunas · {len(arquivos):>2} arquivos")


Cinco cabeçalhos, com **24, 25 e 27 colunas**. Empilhar isso com um `pd.concat` ingênuo
não funciona — e, pior, em alguns casos *funciona sem reclamar*, produzindo lixo.

Vamos ver exatamente onde eles divergem.


In [ ]:
lista = [c.split(";") for c in cabecalhos]
mais_completo = max(lista, key=len)

for i, colunas in enumerate(lista, 1):
    faltando = [c for c in mais_completo if c not in colunas]
    print(f"esquema {i} ({len(colunas)} col.) — falta: {faltando or 'nada'}")


### 1.2 A armadilha: rótulos que mentem

Faltar coluna é o problema fácil — é visível. O problema difícil é que **em parte dos
arquivos o rótulo da coluna não corresponde ao conteúdo dela**.

Veja as três últimas colunas de dois esquemas:


In [ ]:
for i, colunas in enumerate(lista, 1):
    datas_no_cabecalho = [c for c in colunas if "data" in c.lower()]
    print(f"esquema {i}: {datas_no_cabecalho}")


Repare: **`Data Acidente` aparece duas ou três vezes** no mesmo cabeçalho. E não é erro de
transcrição — são colunas distintas do CSV, com conteúdos diferentes. Em um dos esquemas a
coluna rotulada `Data Acidente` repete a *competência*; em outro, a coluna que deveria ser
`Data Emissão CAT` está rotulada `Data Acidente` e repete a data do acidente.

**Consequência prática:** ler os arquivos por nome de coluna mistura conteúdos diferentes
sem gerar erro nenhum. Por isso o leitor deste projeto mapeia **por posição**, ancorado no
cabeçalho reconhecido, e **recusa** cabeçalho desconhecido em vez de adivinhar.


In [ ]:
for leiaute in esquemas.LEIAUTES:
    print(f"{leiaute.nome:<20} {leiaute.n_colunas} colunas · {leiaute.observacao}")


### 1.3 O dicionário oficial está errado sobre as datas

O dicionário publicado declara as datas como `AAAAMMDD`. **Nenhum arquivo usa esse
formato.** Vamos conferir direto nos dados.


In [ ]:
exemplo = sorted(objetos, key=lambda o: o.tamanho)[0].nome
print(f"arquivo: {exemplo}\n")

url = f"https://storage.googleapis.com/acidentes-no-trabalho/{urllib.parse.quote(exemplo)}"
pedido = urllib.request.Request(url, headers={"Range": "bytes=0-4000"})
with urllib.request.urlopen(pedido, timeout=60) as resposta:
    trecho = resposta.read().decode("latin-1")

colunas = trecho.split("\n")[0].strip().split(";")
linha = trecho.split("\n")[1].split(";")
for i, (coluna, valor) in enumerate(zip(colunas, linha, strict=False), 1):
    if "data" in coluna.lower():
        print(f"  col {i:>2}  {coluna:<26} = {valor!r}")

print("\nO dicionário oficial diz:")
for variavel in dicionario.VARIAVEIS:
    if variavel.e_data:
        print(f"  {variavel.rotulo:<26} {variavel.tipo}")


`DD/MM/AAAA`, não `AAAAMMDD`. E `00/00/0000` é o marcador de ausência — não uma data.

Há ainda um segundo formato no mesmo acervo: `AAAA/MM`, para **competência** (o mês de
referência), com `0000/00` como ausência. Os dois convivem no mesmo arquivo.


In [ ]:
amostra = pd.Series(["02/01/2024", "00/00/0000", "29/10/2025", "", "32/13/2024"])
print("como data exata:")
print(datas.para_data(amostra).to_string(), "\n")

competencia = pd.Series(["2022/01", "0000/00", "2020/12"])
print("como competência:")
print(datas.para_competencia(competencia).to_string())


### 1.4 Encoding misto — o erro que não dá erro

Três arquivos estão em UTF-8; os outros 58, em latin-1.

Isso importa porque **latin-1 decodifica qualquer sequência de bytes sem falhar**. Ler um
arquivo UTF-8 como latin-1 não levanta exceção — apenas entrega `EspÃ©cie` no lugar de
`Espécie`, e o erro se propaga silenciosamente até a análise.


In [ ]:
texto = "Espécie do benefício"
bytes_utf8 = texto.encode("utf-8")

print(f"original ......... {texto}")
print(f"lido como latin-1  {bytes_utf8.decode('latin-1')}   ← sem erro, mas corrompido")
print(f"lido como utf-8 .. {bytes_utf8.decode('utf-8')}")


Por isso a detecção testa **UTF-8 estrito primeiro** e só aceita latin-1 quando o texto
não é UTF-8 válido — a ordem inversa nunca falharia e sempre daria a resposta errada.


## 2. Rodando o pipeline

Agora que os problemas estão mapeados, o pipeline resolve todos de uma vez, em quatro
etapas:

```
baixar      bucket GCS             →  data/raw/*.csv
normalizar  data/raw/*.csv         →  data/interim/*.parquet   (unifica os 5 esquemas)
consolidar  data/interim/*.parquet →  data/processed/cat.parquet
relatorio   data/processed         →  reports/relatorio-dados.md
```

> ⏱️ Esta célula baixa 1,8 GB e leva alguns minutos. Para uma execução rápida de
> demonstração, mude `ACERVO_COMPLETO` para `False`: o pipeline roda igual, com 6 arquivos
> (um de cada esquema), mas os números não baterão com os do relatório.


In [ ]:
ACERVO_COMPLETO = True

if ACERVO_COMPLETO:
    caminhos = pipeline.baixar()
else:
    amostra = ["D.SDA.PDA.005.CAT.202501.csv", "D.SDA.PDA.005.CAT.202511.csv",
               "D.SDA.PDA.005.CAT.202211.csv", "D.SDA.PDA.005.CAT.202306.csv",
               "cat-comp01-02-03-2020.csv", "cat-competencia-04-05-06-2020.csv"]
    caminhos = [gcs.baixar(f"cats/{nome}") for nome in amostra]

print(f"{len(caminhos)} arquivos em {DADOS_RAW}")


In [ ]:
%%time
pipeline.normalizar()
caminho_base = pipeline.consolidar()
print(f"\nbase consolidada: {caminho_base.stat().st_size/1e6:.0f} MB")


### 2.1 O que a normalização fez

Vale abrir um arquivo antes e depois. Antes: largura fixa, texto preenchido com espaços,
datas como texto, marcadores de ausência gravados como se fossem valores.


In [ ]:
arquivo = sorted(DADOS_RAW.glob("*.csv"))[0]
print(f"arquivo: {arquivo.name}")
print(f"leiaute reconhecido: {esquemas.identificar_leiaute(arquivo).nome}")
print(f"encoding detectado: {esquemas.detectar_encoding(arquivo)}\n")

cru = pd.read_csv(arquivo, sep=";", encoding=esquemas.detectar_encoding(arquivo),
                  nrows=3, dtype="string")
print("CRU (5 primeiras colunas):")
print(cru.iloc[:, :5].to_string())


In [ ]:
normalizado = esquemas.ler(arquivo, nrows=3)
print("NORMALIZADO:")
print(normalizado[["agente_causador", "cbo_codigo", "data_acidente",
                   "data_nascimento", "leiaute"]].to_string())
print("\ntipos:")
print(normalizado[["data_acidente", "competencia", "sexo"]].dtypes.to_string())


Todos os arquivos, de todos os esquemas, saem com **as mesmas colunas** — as ausentes num
leiaute vêm nulas, em vez de sumir. É isso que permite empilhá-los.


## 3. As armadilhas que só aparecem na base inteira

Algumas coisas não dá para ver num arquivo só.

> 💡 A partir daqui, cada análise carrega **apenas as colunas de que precisa**. São 3,9
> milhões de registros: ler a base inteira várias vezes estoura a memória do Colab. O
> Parquet é colunar, então pedir 3 colunas custa 3 colunas.


In [ ]:
contagem = pipeline.carregar(colunas=["arquivo", "duplicata"])
print(f"linhas nos arquivos: {num(len(contagem))}")


### 3.1 Os arquivos se sobrepõem

A primeira surpresa: uma parte das linhas é **repetição**. Não por erro de leitura — por
como o acervo é publicado.


In [ ]:
duplicadas = int(contagem["duplicata"].sum())
print(f"linhas repetidas: {num(duplicadas)} ({pct(duplicadas, len(contagem))})")

por_arquivo = contagem.groupby("arquivo", observed=True)["duplicata"].agg(["size", "sum"])
por_arquivo["%"] = 100 * por_arquivo["sum"] / por_arquivo["size"]
integrais = por_arquivo[por_arquivo["%"] >= 99.9]
print(f"\narquivos que são republicação INTEGRAL: {len(integrais)}")
print(integrais.sort_values("size", ascending=False)[["size"]].head(8).to_string())


Por que isso acontece? Porque cada arquivo cobre uma **janela de mês de emissão** da CAT,
e as janelas **não são disjuntas**.


In [ ]:
emissoes = pipeline.carregar(colunas=["arquivo", "data_emissao_cat"])
recorte = emissoes[emissoes["arquivo"].str.contains("2022", na=False)]
janelas = recorte.groupby("arquivo", observed=True)["data_emissao_cat"].agg(
    ["min", "max", "size"])
janelas.columns = ["emissão de", "emissão até", "linhas"]
print(janelas.to_string())
del emissoes, recorte


A competência `202207` cobre emissões de julho a novembro de 2022; a `202208` cobre agosto
a novembro — **inteiramente contida na anterior**. Daí o arquivo estar 100% duplicado.

A consolidação **marca** a repetição numa coluna booleana em vez de apagar a linha: a base
continua sendo o registro fiel do acervo, e a decisão fica auditável. Para *contar*
acidentes, usa-se `unicos=True`.


In [ ]:
brutas = len(contagem)
print(f"linhas brutas ... {num(brutas)}")
print(f"registros únicos  {num(brutas - duplicadas)}")
print(f"diferença ....... {num(duplicadas)} — quem empilha sem cuidado superconta isso")
del contagem


### 3.2 A coluna de UF do acidente está corrompida

Esta é a mais séria. A base tem duas colunas de UF: a do **acidente** e a do
**empregador**. Olhe a primeira.


In [ ]:
geo = pipeline.carregar(colunas=["uf_acidente", "uf_empregador_sigla"], unicos=True)
print(geo["uf_acidente"].value_counts(dropna=False).head(12).to_string())


Alguma coisa está muito errada: **Maranhão em primeiro lugar** e **São Paulo ausente**.
Isso contraria tudo que se sabe sobre distribuição de emprego no Brasil.

Para descobrir o que aconteceu, cruzamos com uma UF que sabemos ser confiável: a derivada
do **código IBGE do município** (os dois primeiros dígitos do código são a UF, e o código
nunca vem truncado).


In [ ]:
sub = geo.dropna(subset=["uf_acidente", "uf_empregador_sigla"])
for rotulo, grupo in sub.groupby("uf_acidente", observed=True):
    contagem = grupo["uf_empregador_sigla"].value_counts()
    print(f"  gravado como {rotulo:<18} → na verdade {contagem.index[0]:<3} "
          f"({pct(contagem.iloc[0], len(grupo), 0)} dos {num(len(grupo))} casos)")


Os rótulos estão **sistematicamente trocados**: São Paulo foi gravado como "Maranhão",
Minas Gerais como "Rondônia", Paraná como "Roraima". A concentração é de 94% a 99% — não
é ruído, é um mapeamento errado aplicado na origem.

Poderia-se pensar em recodificar. Mas há um problema maior:


In [ ]:
alvos = {grupo["uf_empregador_sigla"].value_counts().index[0]
         for _, grupo in sub.groupby("uf_acidente", observed=True)}
todas = set(geo["uf_empregador_sigla"].dropna().unique())

print(f"UFs que uf_acidente consegue representar: {len(alvos)}")
print(f"UFs que ela NUNCA representa: {sorted(todas - alvos)}")
nulos = int(geo["uf_acidente"].isna().sum())
print(f"\nregistros sem rótulo algum: {pct(nulos, len(geo))} — "
      "é onde essas UFs foram parar")


**Doze UFs não têm rótulo nenhum** — entre elas Rio Grande do Sul, Santa Catarina, Bahia e
Goiás. Elas colapsam todas no mesmo marcador de ausência. A informação simplesmente não
está lá, e nenhuma recodificação a traz de volta.

**Decisão:** descartar a coluna na análise e usar a UF derivada do código do município,
registrando que ela localiza **o empregador, não o acidente**.


In [ ]:
sem_uf = limpeza.descartar_colunas_nao_confiaveis(geo)
print("colunas removidas:", [c for c in geo.columns if c not in sem_uf.columns])
del geo, sub, sem_uf


### 3.3 O marcador de ausência vem truncado

Os campos têm largura fixa. Isso corta as descrições longas — e corta **também o próprio
marcador de ausência**, que aparece em pedaços diferentes conforme a coluna.

Foi um bug real deste projeto: a primeira versão da limpeza casava o texto inteiro
(`{ñ class}`) e deixava passar as versões cortadas, 90.844 registros contados como se
fossem uma categoria válida.


In [ ]:
for variante in ["{ñ class}", "{ñ class", "{ñ", "{"]:
    serie = pd.Series([variante], dtype="string")
    resultado = limpeza.marcar_sentinelas(pd.DataFrame({"x": serie}))["x"][0]
    print(f"  {variante!r:<14} → {'ausência' if pd.isna(resultado) else repr(resultado)}")

print("\nE o que NÃO é marcador:")
for texto in ["Classificador de Graos", "Zerador"]:
    serie = pd.Series([texto], dtype="string")
    resultado = limpeza.marcar_sentinelas(pd.DataFrame({"x": serie}))["x"][0]
    print(f"  {texto!r:<26} → {'ausência' if pd.isna(resultado) else 'preservado'}")


A regra usada: **qualquer valor que comece com `{` é ausência**. Foi possível adotá-la
porque se verificou, nos 3,9 milhões de registros, que nenhum valor legítimo começa com
chave — e essa regra resiste a qualquer truncamento.

### 3.4 O truncamento fragmenta categorias

Consequência mais sutil do mesmo problema: se a descrição é cortada em 20 caracteres, o
**mesmo setor econômico aparece sob dois rótulos**.


In [ ]:
rotulos = pipeline.carregar(
    colunas=["cnae_codigo", "cnae_descricao",
             "codigo_municipio_empregador", "nome_municipio_empregador"],
    unicos=True,
)

for codigo, descricao in [("cnae_codigo", "cnae_descricao"),
                          ("codigo_municipio_empregador", "nome_municipio_empregador")]:
    pares = rotulos.dropna(subset=[codigo, descricao])
    por_codigo = pares.groupby(codigo, observed=True)[descricao].nunique()
    fragmentados = int((por_codigo > 1).sum())
    print(f"{codigo:<28} {por_codigo.size:>5} códigos COM descrição · "
          f"{fragmentados:>4} deles ({pct(fragmentados, por_codigo.size)}) "
          "aparecem sob mais de um rótulo")

exemplo = rotulos.dropna(subset=["cnae_codigo", "cnae_descricao"])
multi = exemplo.groupby("cnae_codigo", observed=True)["cnae_descricao"].unique()
for codigo, descricoes in multi.items():
    if len(descricoes) > 1:
        print(f"\nexemplo — CNAE {codigo}:")
        for d in descricoes:
            print(f"   {d!r}")
        break
del rotulos, exemplo, multi


**Agrupar pela descrição, portanto, está errado.** Todo agrupamento neste projeto usa o
**código**; a descrição serve apenas como rótulo de exibição, tomando-se a versão mais
longa observada para cada código.

Isso não é detalhe: refazer o ranking de letalidade por setor pelo código mudou o primeiro
colocado de 1,70% para 1,96%.


## 4. Análise exploratória

Com a base consolidada e as armadilhas contornadas, dá para analisar.


In [ ]:
COLUNAS_ANALISE = [
    "data_acidente", "data_emissao_cat", "ano_acidente", "mes_acidente",
    "sexo", "idade_acidente", "tipo_acidente", "indica_obito",
    "cnae_codigo", "cnae_descricao", "parte_corpo_atingida",
    "uf_empregador_sigla", "codigo_municipio_empregador",
]
analise = pipeline.carregar(colunas=COLUNAS_ANALISE, unicos=True)

print(f"registros únicos: {num(len(analise))}")
periodo = analise["data_acidente"].dropna()
print(f"período: {periodo.min():%m/%Y} a {periodo.max():%m/%Y}")
obitos = int((analise["indica_obito"] == "Sim").sum())
print(f"óbitos registrados: {num(obitos)} ({pct(obitos, len(analise), 3)})")


### 4.1 O volume ao longo do tempo — e por que ele não pode ser lido ingenuamente


In [ ]:
figuras.serie_mensal(analise)
display(Image("reports/figuras/serie-mensal.png"))


A série **não é utilizável como está**. Ela oscila entre 8 mil e 128 mil por mês, sem
padrão epidemiológico possível. A causa é a cobertura irregular: não existe arquivo
cobrindo as emissões de janeiro e fevereiro de 2022, e a defasagem mediana entre acidente
e emissão é de apenas 3 dias — logo, os acidentes desses meses estão majoritariamente
**ausentes**.

Dois sinais, porém, se distinguem do ruído: a **queda de abril de 2020**, compatível com o
início da pandemia, e o **pico de julho de 2025**, sem explicação no fenômeno.


In [ ]:
defasagem = (analise["data_emissao_cat"] - analise["data_acidente"]).dt.days.dropna()
print("dias entre o acidente e a emissão da CAT:")
print(defasagem.describe(percentiles=[.5, .9, .99]).round(1).to_string())


### 4.2 Quem se acidenta


In [ ]:
print((100*analise["sexo"].value_counts(normalize=True)).round(2).to_string())
print()
print("idade no momento do acidente:")
print(analise["idade_acidente"].describe(percentiles=[.25, .5, .75]).round(1).to_string())


In [ ]:
figuras.idade_por_sexo(analise)
display(Image("reports/figuras/idade-por-sexo.png"))


As distribuições diferem em **forma**, não apenas em posição: os homens concentram-se entre
22 e 28 anos, enquanto as mulheres formam um platô mais alto e mais tardio, entre 30 e 45.
Isso sugere composição ocupacional distinta, não apenas idades médias diferentes.


### 4.3 Que tipo de acidente é mais letal


In [ ]:
com_obito = analise[analise["indica_obito"].notna()]
tabela = pd.crosstab(com_obito["tipo_acidente"], com_obito["indica_obito"])
tabela["letalidade"] = [
    pct(sim, sim + nao, 3)
    for sim, nao in zip(tabela["Sim"], tabela["Não"], strict=True)
]
tabela["ordem"] = tabela["Sim"] / (tabela["Sim"] + tabela["Não"])
print(tabela.sort_values("ordem", ascending=False).drop(columns="ordem").to_string())


In [ ]:
figuras.letalidade_por_tipo(analise)
display(Image("reports/figuras/letalidade-por-tipo.png"))


**Inversão relevante.** O acidente de **trajeto** é o menos frequente entre os dois
principais (22% contra 74%), mas o **mais letal**: 0,88% contra 0,33% — 2,7 vezes maior.
Em números absolutos são 6.743 óbitos em trajeto contra 8.421 em típicos, apesar de o
típico ser 3,3 vezes mais frequente.


### 4.4 Que setores concentram a letalidade


In [ ]:
figuras.letalidade_por_setor(analise)
display(Image("reports/figuras/letalidade-por-setor.png"))

mortes = int((com_obito["indica_obito"] == "Sim").sum())
print(f"letalidade geral da base: {pct(mortes, len(com_obito), 3)}")


**Transporte Rodoviário de Carga** lidera com 1,96% — mais de **4 vezes a média geral**.
Os demais setores do topo também envolvem via pública ou veículos.

Somado ao achado anterior, os dois apontam para o mesmo mecanismo: **a exposição ao
trânsito**. É a hipótese mais promissora que a EDA produziu.


### 4.5 Onde estão os empregadores, e o que foi atingido


In [ ]:
figuras.registros_por_uf(analise)
display(Image("reports/figuras/registros-por-uf.png"))


In [ ]:
figuras.parte_do_corpo(analise)
display(Image("reports/figuras/parte-do-corpo.png"))


## 5. Síntese

### O que a consolidação exigiu

| Problema | Como se manifesta | Decisão |
|---|---|---|
| 5 esquemas diferentes | 24, 25 e 27 colunas | Mapear **por posição**, recusar cabeçalho desconhecido |
| Rótulos que mentem | `Data Acidente` repetido, apontando conteúdos distintos | Descartar as posições cujo rótulo não corresponde |
| Formato de data | Dicionário diz `AAAAMMDD`; arquivos usam `DD/MM/AAAA` e `AAAA/MM` | Conversor próprio, sentinelas viram nulo |
| Encoding misto | 3 arquivos UTF-8, 58 latin-1 — e latin-1 nunca falha | Testar UTF-8 estrito primeiro |
| Marcador truncado | `{ñ class}`, `{ñ class`, `{ñ` | Qualquer valor iniciado por `{` é ausência |
| Sobreposição de arquivos | 11,7% de republicação; 8 arquivos redundantes | **Marcar**, não apagar; `unicos=True` para contar |
| UF do acidente corrompida | Rótulos trocados; 12 UFs sem rótulo | Descartar; usar UF do código IBGE |
| Descrição truncada | 84,7% dos CNAE com mais de um rótulo | Agrupar por **código** |

### Os três principais achados

1. **O acervo superconta 11,7%** se os arquivos forem empilhados, e sua cobertura temporal
   é irregular. Isso determina o que é possível analisar.
2. **O acidente de trajeto é 2,7× mais letal que o típico**, e o transporte rodoviário de
   carga é o setor mais letal da base — dois caminhos independentes para o mesmo mecanismo.
3. **A UF do acidente é irrecuperável**: a geografia só pode ser analisada pelo empregador.

### As limitações que restam

- **Sem denominador de exposição.** A base registra acidentes, não trabalhadores expostos.
  O que se chama aqui de "letalidade" é a proporção de óbitos *entre os acidentes
  comunicados* — não a probabilidade de morrer no setor. Cruzar com RAIS ou CAGED por CNAE
  permitiria estimar risco de verdade.
- **Subnotificação.** Só há acidentes **comunicados**, e a propensão a comunicar
  provavelmente varia entre setores e portes de empresa.
- **Sem identificador de registro.** A detecção de republicação compara o conteúdo
  integral da linha; duas CATs realmente distintas e idênticas em todos os campos seriam
  contadas como uma só.
- **Localização é do empregador.** Acidentes de trajeto e trabalho em campo ocorrem longe
  da sede.

### Reproduzir

```bash
git clone https://github.com/mrcsvg/ufpr-ppgcd-acidentes-no-trabalho.git
cd ufpr-ppgcd-acidentes-no-trabalho
make setup && make dados
```

O relatório descritivo completo é regerado por `make relatorio`; as armadilhas estão
documentadas em `docs/qualidade-dos-dados.md`.
